# Leakage-Aware Parkinson’s Voice Classification — Google Colab

Notebook này tái lập đúng quy trình canonical của repository: dữ liệu bảng gồm 22
đặc trưng acoustic được kiểm tra, loại 2 đặc trưng dư thừa còn 20 đặc trưng cố định,
sau đó đánh giá bằng nested stratified subject CV (4 outer × 3 inner). Production chỉ
dùng `StandardScaler → LogisticRegression`, gộp median ở cấp subject và chọn threshold
từ OOF train.

Đây là prototype nghiên cứu/sàng lọc, không phải công cụ chẩn đoán. Chọn **Runtime → Run all**.

## 1. Khóa môi trường chạy

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
PACKAGES = [
    "pandas==2.2.3", "numpy==2.1.3", "scikit-learn==1.7.1",
    "joblib==1.4.2", "matplotlib==3.9.2",
]
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
os.environ.setdefault("MPLBACKEND", "Agg")
print("Môi trường:", "Google Colab" if IN_COLAB else "Python cục bộ")

## 2. Khôi phục dự án tối thiểu

Notebook tự chứa dữ liệu, cấu hình và toàn bộ package `parkinson_voice` cần cho audit, train và
inference. Không phụ thuộc đường dẫn Windows, Google Drive hoặc GitHub.

In [ ]:
import base64, io, shutil, zipfile
from pathlib import Path

PAYLOAD = "..."
PROJECT_DIR = (
    Path("/content/parkinsons-voice-classification")
    if IN_COLAB
    else Path.cwd() / "parkinsons-colab-runtime"
)
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Thư mục chạy:", PROJECT_DIR)

## 3. Kiểm tra phiên bản, checksum và schema

In [ ]:
import json, joblib, numpy as np, pandas as pd, sklearn
from parkinson_voice.data import ORIGINAL_FEATURES, SUBJECT_COLUMN, TARGET_COLUMN, load_data
from parkinson_voice.features import MODEL_FEATURES, REDUNDANT_FEATURES
from parkinson_voice.utils import sha256_file

expected_versions = {
    "pandas": "2.2.3", "numpy": "2.1.3", "scikit-learn": "1.7.1",
    "joblib": "1.4.2",
}
actual_versions = {
    "pandas": pd.__version__, "numpy": np.__version__, "scikit-learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
if IN_COLAB:
    assert actual_versions == expected_versions, (actual_versions, expected_versions)
DATA_PATH = Path("data/parkinsons.csv")
DATA_SHA256 = "32e6040916d2f5b80b49589d925a92bd25420687c76be19d72e37205e104abe6"
assert sha256_file(DATA_PATH) == DATA_SHA256
frame = load_data(DATA_PATH)
assert len(frame) == 195 and frame[SUBJECT_COLUMN].nunique() == 32
assert len(ORIGINAL_FEATURES) == 22 and len(MODEL_FEATURES) == 20
assert set(REDUNDANT_FEATURES) == {"Jitter:DDP", "Shimmer:DDA"}
assert set(frame[TARGET_COLUMN].unique()) == {0, 1}
print(f"✅ {len(frame)} recordings | 32 subjects | 22 source features | 20 model features")
display(frame.head(3))

## 4. Audit phân chia theo subject

In [ ]:
from parkinson_voice.audit import build_data_manifest
from parkinson_voice.evaluate import make_subject_folds

manifest = build_data_manifest(frame, DATA_PATH)
assert manifest["duplicate_recording_names"] == 0
assert manifest["duplicate_full_feature_vectors"] == 0
folds = make_subject_folds(frame, n_splits=4, random_state=42)
for number, (fit_index, valid_index) in enumerate(folds, 1):
    fit_ids = set(frame.iloc[fit_index][SUBJECT_COLUMN])
    valid_ids = set(frame.iloc[valid_index][SUBJECT_COLUMN])
    assert fit_ids.isdisjoint(valid_ids)
    assert frame.iloc[valid_index][TARGET_COLUMN].nunique() == 2
    print(f"Fold {number}: subject disjoint, validation đủ hai lớp")
print("✅ Không có overlap subject trong outer CV")

## 5. Huấn luyện canonical và sinh artifact

In [ ]:
from parkinson_voice.train import train

ARTIFACT_DIR = Path("artifacts")
comparison = train(DATA_PATH, ARTIFACT_DIR)
display(comparison)

## 6. Kiểm tra kết quả và artifact release

In [ ]:
from parkinson_voice.predict import load_bundle

EXPECTED = json.loads("{\n  \"dataset\": {\n    \"dataset\": \"UCI Parkinsons\",\n    \"dataset_sha256\": \"32e6040916d2f5b80b49589d925a92bd25420687c76be19d72e37205e104abe6\",\n    \"n_recordings\": 195,\n    \"n_subjects\": 32,\n    \"subject_distribution\": {\n      \"0\": 8,\n      \"1\": 24\n    },\n    \"source_features\": 22,\n    \"model_features\": 20,\n    \"subject_id_rule\": \"drop_final_recording_suffix\",\n    \"dropped_features\": [\n      \"Jitter:DDP\",\n      \"Shimmer:DDA\"\n    ],\n    \"duplicate_recording_names\": 0,\n    \"duplicate_full_feature_vectors\": 0,\n    \"recordings_per_subject\": {\n      \"min\": 6,\n      \"median\": 6.0,\n      \"max\": 7\n    },\n    \"evaluation_protocol\": \"nested-stratified-subject-cv-4x3\"\n  },\n  \"selection\": {\n    \"C\": 0.01,\n    \"class_weight\": \"balanced\"\n  },\n  \"nested_cv_subject\": {\n    \"Accuracy\": 0.6875,\n    \"Balanced Accuracy\": 0.625,\n    \"Precision\": 0.8181818181818182,\n    \"Recall/Sensitivity\": 0.75,\n    \"Specificity\": 0.5,\n    \"NPV\": 0.4,\n    \"F1-macro\": 0.6135265700483092,\n    \"ROC-AUC\": 0.7395833333333333,\n    \"Brier score\": 0.21462450560246463,\n    \"fold_mean\": {\n      \"Balanced Accuracy\": 0.625,\n      \"F1-macro\": 0.5870629370629371,\n      \"ROC-AUC\": 0.75\n    },\n    \"fold_std\": {\n      \"Balanced Accuracy\": 0.15023130314433286,\n      \"F1-macro\": 0.13159179156242765,\n      \"ROC-AUC\": 0.15590239111558088\n    }\n  },\n  \"deployment_oof\": {\n    \"Accuracy\": 0.8125,\n    \"Balanced Accuracy\": 0.8333333333333333,\n    \"Precision\": 0.95,\n    \"Recall/Sensitivity\": 0.7916666666666666,\n    \"Specificity\": 0.875,\n    \"NPV\": 0.5833333333333334,\n    \"F1-macro\": 0.7818181818181817,\n    \"ROC-AUC\": 0.875,\n    \"Brier score\": 0.20014339554667712,\n    \"ECE (5 bins)\": 0.27497968338873324,\n    \"decision_threshold\": 0.4206085668611331,\n    \"aggregation\": \"median\"\n  },\n  \"evaluation_protocol\": {\n    \"outer_folds\": 4,\n    \"inner_folds\": 3,\n    \"unit\": \"subject\",\n    \"primary_metric\": \"Balanced Accuracy\"\n  },\n  \"artifact\": \"releases/v1.0.0/model.joblib\"\n}")
ACTUAL = json.loads((ARTIFACT_DIR / "metrics.json").read_text(encoding="utf-8"))
assert ACTUAL["dataset"]["dataset_sha256"] == EXPECTED["dataset"]["dataset_sha256"]
assert ACTUAL["dataset"]["n_recordings"] == 195
assert ACTUAL["dataset"]["n_subjects"] == 32
assert ACTUAL["selection"] == EXPECTED["selection"]
for metric in ("Balanced Accuracy", "F1-macro", "ROC-AUC"):
    assert np.isclose(
        ACTUAL["nested_cv_subject"][metric],
        EXPECTED["nested_cv_subject"][metric],
        rtol=0,
        atol=1e-12,
    )
bundle = load_bundle(ARTIFACT_DIR / "releases" / "v1.0.0" / "model.joblib")
assert bundle["feature_columns"] == MODEL_FEATURES
assert bundle["aggregation"] == "median"
print("✅ Kết quả canonical và release model khớp repository")
print(json.dumps(ACTUAL, ensure_ascii=False, indent=2))

## 7. Suy luận dữ liệu mới — tùy chọn

In [ ]:
from parkinson_voice.predict import load_bundle, predict_records

inference_frame = frame.drop(columns=[TARGET_COLUMN, SUBJECT_COLUMN])
record_results, subject_results = predict_records(inference_frame, bundle)
assert "screening_score" in record_results
assert "subject_screening_score" in subject_results
display(subject_results.head())
print("Input inference chỉ dùng name và feature; không truyền status.")

## Kết luận

Notebook hoàn tất khi tất cả assertion màu xanh: checksum/schema đúng, outer CV không
overlap subject, kết quả nested CV khớp artifact và release model dùng đúng 20-feature
contract. Các score chỉ là tín hiệu nghiên cứu; cần cohort ngoài và validation lâm sàng
trước mọi diễn giải y tế.